# IRC Plots

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4/blob/main/book/10-Solvent/103-IRCPlots.ipynb)
:::

As discussed in the [previous notebook](102-IRCData.ipynb), we now have **two data files** with the gas phase and solvent IRC calculation data.

| Files | Description |
| :--------------------------- | :------------------------- |
| `sn2_irc_water_B3LYP321G.json` | The solvent phase (water, PCM model) IRC calculation |
| `sn2_irc_gas_B3LYP321G.json` | The gas phase IRC calculation |

We can now make some **comparisons**.

This notebook does not run *Psi4*, so the hidden first cell only imports *NumPy*, *pandas* and *Matplotlib* and applies the plot style sheet.

In [ ]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plot style sheets from the book's GitHub repository
if IN_COLAB and not Path("styles/tufte.mplstyle").exists():
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/tufte.mplstyle
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/S2_classic2.mplstyle

if IN_COLAB:
    plt.style.use("styles/tufte.mplstyle")
else:
    plt.style.use("../../styles/tufte.mplstyle")

## Importing Stored Data

No need to use the raw data and process the files. We have **stored** that work as data files and **documented** the process that we used to clean and process the data. Now we can load them in and use them. As before, if the files are not in this folder we download my copies from the book's *GitHub* repository.

In [ ]:
url = "https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/book/10-Solvent/"

def read_json(filename):
    if Path(filename).exists():
        return pd.read_json(filename)
    return pd.read_json(url + filename)

solvent = read_json("sn2_irc_water_B3LYP321G.json")
gasphase = read_json("sn2_irc_gas_B3LYP321G.json")

print("Solvent IRC Data")
display(solvent.head())
print("\nGas Phase IRC Data")    # the '\n' is a 'new line' character
display(gasphase.head())

| Line | Notes |
| :---- | :---- |
| `pd.read_json(filename)` | Reads a JSON file of records straight into a dataframe, column names and all. It works with a file name or a web address. |
| `print("\nGas Phase IRC Data")` | The `\n` is a "new line" character, which leaves a blank line before the title. |

## Plots

First we must **decide** what to plot. The stride value should be similar for each since it is the same bonds in both cases that are changing (stride is a composite value that encompasses the totality of bond changes). How do the energies **compare** in gas phase vs. water as bonds break and form across the transition state? Let us plot the **relative energy** vs. **stride**.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,5))

ax.plot(solvent["stride"], solvent["E(kJ/mol)"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.5,
        label="water")
ax.plot(gasphase["stride"], gasphase["E(kJ/mol)"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="#FFB3BA", markeredgecolor="black", markeredgewidth=0.5,
        label="gas phase")

ax.set(title="Intrinsic Reaction Coordinate",
       ylabel=r"Energy $\left(\frac{kJ}{mol}\right)$",
       xlabel=r"Stride $\left(Bohr \cdot \sqrt{amu}\right)$")
ax.legend()

#fig.savefig("plot1.pdf")
plt.show()

| Line | Notes |
| :---- | :---- |
| `ax.plot(solvent["stride"], solvent["E(kJ/mol)"], ...)` | The water IRC: open white circles joined by a thin black line. |
| `markerfacecolor="#FFB3BA"` | The gas phase IRC gets pastel pink circles. |
| `label="water"` and `ax.legend()` | Each series gets a name, and `legend()` draws the key. |
| `ax.set(title=..., ylabel=..., xlabel=...)` | Sets the title and axis labels at once. Text between `$` signs is typeset as math. |
| `#fig.savefig("plot1.pdf")` | Remove the `#` to save the figure as a PDF file. |

[[Range of relative energies at the ends of each path; number of points; did the gas path stop early?]]

In the gas phase, the **relative energy** between the ground state and the transition state is **much smaller**. This is because the **negatively charged** chloride nucleophile/leaving group is very **high in energy** in a **vacuum** where it cannot share its electrons. In the **solvent** model, the energy is less as chloride can be stabilized by the **electric dipole** of water. In fact the gas phase IRC ends early because the **energy** is actually about to start **going back up** as the nucleophile moves away. In a vacuum, chloride separated widely from the methyl chloride is actually **higher in energy** than the transition state. [[Check this against the Psi4 result.]]

## Changing Bond Lengths

The famous More O'Ferrall&ndash;Jencks plot follows one **coordinate** of the reaction with respect to **another**. This reaction has a **nucleophile** that moves in to make a bond and a **leaving group** that departs, breaking a bond. We could plot "making" on one axis and "breaking" on the other.

There are **two** C&ndash;Cl distances in each IRC: the **incoming** nucleophile and the departing leaving group. The C&ndash;Cl bond in methyl chloride is calculated to be about [[CH3Cl C–Cl from 101]]&nbsp;&Aring;. The incoming nucleophile starts at a longer distance and **moves** in. The leaving group bond will lengthen. At the transition state, they should be identical (defined by symmetry). Let us plot **one** length against the **other**. Will it be a straight line? Will it be a curve?

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5,5))

low = min(solvent["C-Cl1"].min(), solvent["C-Cl2"].min()) - 0.05
high = max(solvent["C-Cl1"].max(), solvent["C-Cl2"].max()) + 0.05
ax.plot([low, high], [low, high], color="lightgray", linewidth=0.5, zorder=1)

ax.plot(solvent["C-Cl2"], solvent["C-Cl1"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.5,
        label="water", zorder=2)
ax.plot(gasphase["C-Cl2"], gasphase["C-Cl1"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="#FFB3BA", markeredgecolor="black", markeredgewidth=0.5,
        label="gas phase", zorder=2)

ax.set(title="Intrinsic Reaction Coordinate",
       ylabel=r"1,3 C–Cl Bond Length $\left(\AA\right)$",
       xlabel=r"2,3 C–Cl Bond Length $\left(\AA\right)$",
       xlim=[low, high],
       ylim=[low, high])
ax.set_aspect("equal")
ax.legend()

#fig.savefig("plot2.pdf")
plt.show()

| Line | Notes |
| :---- | :---- |
| `low = min(...) - 0.05` | The shortest bond length in the water data, less a little margin. `high` is the longest plus a margin. Both axes will use the same range. |
| `ax.plot([low, high], [low, high], ...)` | A light gray **diagonal** line where the two bonds are equal. The transition state sits on it. `zorder=1` puts it behind the data. |
| `ax.set_aspect("equal")` | One &Aring;ngstrom is the same length on both axes, so the plot is square and the diagonal is at 45&deg;. |

The black line is the **solvent** IRC. What would you call that? I vote for "double ended hockey stick". [[Check: does the Psi4 water path show the same shape? At the start, is most of the stride due to the nucleophile moving in? At which distance does the leaving group start to move?]] In the *GAMESS* calculation it was clear that, at the beginning, **most** of the stride score was due to the nucleophile **moving in** towards the carbon. When it had moved in from 3.1&nbsp;&Aring; to about 2.85&nbsp;&Aring; we finally saw the leaving group **start to move** in concert. Would you say that the reaction "begins" when the nucleophile **reaches** that distance? Does it end when the leaving group reaches that distance on the **other side**? Or should we include the translation of the nucleophile in from 3.1&nbsp;&Aring; (or even farther)?

The **gas phase** IRC is shorter because of the high cost of moving the chloride away in a vacuum. The key takeaway here is that solvent modeling has a **very large effect** in charged systems and probably a significant effect in every system.

## Comparing with *GAMESS*

How does our water IRC compare with the one from the *GAMESS* book? The *GAMESS* table is still on *GitHub*, so let us plot the two together. The total energies differ (different versions of B3LYP, and slightly different cavities), but the **relative** energies should be close.

In [ ]:
gamess_url = "https://raw.githubusercontent.com/blinkletter/UsingGAMESS/main/data/SN2_IRC_Solvent_Data.txt"
gamess = pd.read_csv(gamess_url, index_col=0)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,5))

ax.plot(gamess["stride"], gamess["E(kJ/mole)"], color="black", linewidth=0.5, marker="o",
        markersize=3, markerfacecolor="#AEC6CF", markeredgecolor="black", markeredgewidth=0.5,
        label="GAMESS", zorder=1)
ax.plot(solvent["stride"], solvent["E(kJ/mol)"], color="black", linewidth=0.7, marker="o",
        markersize=5, markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.5,
        label="Psi4", zorder=2)

ax.set(title="IRC in water",
       ylabel=r"Energy $\left(\frac{kJ}{mol}\right)$",
       xlabel=r"Stride $\left(Bohr \cdot \sqrt{amu}\right)$")
ax.legend()
plt.show()

| Line | Notes |
| :---- | :---- |
| `pd.read_csv(gamess_url, index_col=0)` | The *GAMESS* table was saved as a comma-separated text file, with the row numbers in the first column. |
| `gamess["E(kJ/mole)"]` | I spelled the column name differently back then. |

[[Compare the two water curves: depth at the ends (GAMESS about −23.2 kJ/mol at ±4.9), shape, point spacing.]]

## Exploring the 3<sup>rd</sup> Dimension

We can use a 3-D plot to combine the information in the two previous plots. Below is a 3-D plot of the **solvent** IRC, with its "shadows" drawn in gray on the floor and two walls of the box. I found some [instructions](https://jakevdp.github.io/PythonDataScienceHandbook/04.12-three-dimensional-plotting.html) in another useful online book, and more [here](https://matplotlib.org/stable/tutorials/toolkits/mplot3d.html).

In [ ]:
x = solvent["C-Cl1"]
y = solvent["C-Cl2"]
z = solvent["E(kJ/mol)"]

low = min(x.min(), y.min())
high = max(x.max(), y.max())

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(6,6))
ax.view_init(15, -60)

ax.plot(x, y, zs=z.min(), zdir="z", color="lightgray", linewidth=0.7)
ax.plot(y, z, zs=low, zdir="x", color="lightgray", linewidth=0.7)
ax.plot(x, z, zs=high, zdir="y", color="lightgray", linewidth=0.7)
ax.plot(x, y, z, color="black", linewidth=0.7)

ax.set_xlabel(r"C3–Cl1 Distance $\left( \AA \right)$")
ax.set_ylabel(r"C3–Cl2 Distance $\left( \AA \right)$")
ax.set_zlabel(r"$\Delta E \  \left(kJ/mol\right)$")
ax.set_title("Intrinsic Reaction Coordinate")
ax.set_xlim(low, high)
ax.set_ylim(low, high)
ax.set_zlim(z.min(), 1)

#fig.savefig("plot3.pdf")
plt.show()

| Line | Notes |
| :---- | :---- |
| `subplot_kw={"projection": "3d"}` | Asks `plt.subplots()` for a 3-D set of axes instead of a flat one. |
| `ax.view_init(15, -60)` | Looks at the box from 15&deg; above the floor, turned &minus;60&deg; around the vertical axis. Change these and see. |
| `ax.plot(x, y, zs=z.min(), zdir="z", ...)` | The **shadow on the floor**: the More O'Ferrall&ndash;Jencks plot, drawn flat at the lowest energy. |
| `ax.plot(y, z, zs=low, zdir="x", ...)` | The shadow on the wall at the lowest C3&ndash;Cl1 distance: energy vs. C3&ndash;Cl2. With `zdir="x"`, the two lists we give are drawn along the *y* and *z* axes. |
| `ax.plot(x, z, zs=high, zdir="y", ...)` | The shadow on the back wall: energy vs. C3&ndash;Cl1. |
| `ax.plot(x, y, z, ...)` | The path itself, in three dimensions. |

Does that help you to imagine the nucleophile translating in, the making and breaking of bonds, and the leaving group translating away? I wonder what other kinds of plots we could use.

## Summary

In the gas phase, the IRC is short and shallow, and the "reactant" at each end is a chloride stuck to the back of a methyl chloride. In water, the chloride is free to leave and the path runs downhill much farther. Should we **redo everything** that we did up until now by switching on the PCM options and running the calculations again? That will be left as an exercise for the reader.

```{warning} AI Statement
:class: dropdown
This page is an adaptation of the second "Solvent IRC Plots" notebook of my [previous web book on using *GAMESS*](https://blinkletter.github.io/UsingGAMESS). About [[half]] of the text is **my own work**, carried over word for word or lightly edited, including the introduction, the discussion of the energy profiles, the More O'Ferrall&ndash;Jencks plot and its "double ended hockey stick", and the 3-D plot. The **rest of the page** was written by an **LLM system** at my request. I asked it to load the JSON files written by the *Psi4* data notebook, to restyle the plots, and to add a comparison with my *GAMESS* results. The *Python* code, the line-by-line explanations of it and the commentary on the results were written by the LLM. The numbers quoted in the text come from my own calculations.
```